# Integrated Experiment Runner

This notebook consolidates the major experiments used in the Adaptive Swarm benchmarking project.

The goal is to avoid maintaining many fragmented notebooks. Dataset-level differences, such as universe, time horizon, and pre/post-normalisation settings, are handled through configuration. Label EDA, feature-set selection, model fitting, ranking evaluation, subgroup diagnostics, temporal diagnostics, and Excel exports are handled by one consistent experiment pipeline.

**Main design principles**

1. Preserve EDA for each label before modelling.
2. Reuse shared modelling and metric functions whenever possible.
3. Keep variable names consistent across the full notebook.
4. Explain each section before the code.
5. Export every major result table to Excel.
6. End with a checklist of tasks completed by this notebook.

## 1. Imports, global settings, and output folders

In [43]:
# import the required libraries
import os
import json
import math
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score, average_precision_score,
    brier_score_loss
)

warnings.filterwarnings('ignore')

# defines global variables used throughout the notebook, capital variables --> global constants / configuration
RANDOM_STATE = 42
N_JOBS = -1

DATE_COL = 'attr__timestamp'
TICKER_COL = 'attr__ticker'
SIC2_COL = 'attr__sic2'
YEAR_COL = 'year'
SPLIT_COL = 'split'

TRUE_COL = 'y_true'
PRED_COL = 'y_pred'
SCORE_COL = 'prediction_score'

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'integrated_experiment_outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
MODEL_DIR = OUTPUT_DIR / 'models'
for d in [OUTPUT_DIR, TABLE_DIR, PREDICTION_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print('Output directory:', OUTPUT_DIR.resolve())

Output directory: C:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\integrated_experiment_outputs


## 2. Dataset configuration

This section defines where the train, validation, and test files are located. Only the dataset paths should need to change when switching between Universe, time horizon, or pre/post-normalisation versions. The rest of the notebook runs from the same `DATASET_CONFIG` structure.

In [7]:
UNIVERSE = "universe_100"
TIME_HORIZON = "recent"  
NORMALISATION_STATUS = "post_normalisation"
# This section defines where the train, validation, and test files are located. 
# When switching between universe, time horizon, or pre/post-normalisation versions, only need to change the value of UNIVERSE/ TIME_HORIZON/ NORMALISATION_STATUS
DATASET_CONFIG = {
    "dataset_name": f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}",
    "train_path": PROJECT_ROOT / f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_train.jsonl",
    "valid_path": PROJECT_ROOT / f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_validation.jsonl",
    "test_path":  PROJECT_ROOT / f"{UNIVERSE}_{TIME_HORIZON}_{NORMALISATION_STATUS}_test.jsonl",
    "project_b_feature_file": PROJECT_ROOT / "feature_project_b.txt",
    "universe": UNIVERSE,
    "time_horizon": TIME_HORIZON,
    "normalisation_setting": NORMALISATION_STATUS,
}
# Check whether all configured files exist
for key in ["train_path", "valid_path", "test_path", "project_b_feature_file"]:
    path = Path(DATASET_CONFIG[key])
    print(f"{key}: {path}")
    print("Exists:", path.exists())
    print("-" * 80)

train_path: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_train.jsonl
Exists: True
--------------------------------------------------------------------------------
valid_path: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_validation.jsonl
Exists: True
--------------------------------------------------------------------------------
test_path: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_test.jsonl
Exists: True
--------------------------------------------------------------------------------
project_b_feature_file: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\feature_project_b.txt
Exists: True
--------------------------------------------------------------------------------


## 3. Data loading utilities

This section loads and prepares the data. It supports flattened parquet/csv files and nested jsonl/json files. The downstream EDA and modelling sections always receive the same structure: one DataFrame per split in the `frames` dictionary.

In [10]:
def load_flatten_jsonl(path):
    """
    Load and flatten an Adaptive Swarm JSONL dataset.

    Expected JSONL structure:
    {
        "section": "data",
        "data": {
            "IndexReference": ...,
            "Attributes": {...},
            "Features": {...},
            "Labels": {...}
        }
    }

    This function extracts raw_record["data"] and flattens:
    - Attributes / attributes / attrs -> attr__
    - Features / features             -> feature__
    - Labels / labels                 -> label__

    Each JSONL line becomes one row in the returned DataFrame.
    """
    rows = []
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            raw_record = json.loads(line)

            # Keep only actual data rows.
            if raw_record.get("section") != "data":
                continue

            # The useful content is stored inside raw_record["data"].
            record = raw_record.get("data", {})

            if not isinstance(record, dict):
                continue

            attrs = (record.get("Attributes"))
            feats = (record.get("Features"))
            labs = (record.get("Labels"))

            row = {
                "IndexReference": record.get("IndexReference")
            }

            for k, v in attrs.items():
                row[k if str(k).startswith("attr__") else f"attr__{k}"] = v

            for k, v in feats.items():
                row[k if str(k).startswith("feature__") else f"feature__{k}"] = v

            for k, v in labs.items():
                row[k if str(k).startswith("label__") else f"label__{k}"] = v

            # Preserve simple scalar fields inside data, such as Markers.
            for k, v in record.items():
                if (
                    k not in ["Attributes","Features","Labels"]
                    and not isinstance(v, (dict, list))
                ):
                    row.setdefault(k, v)

            # Preserve top-level scalar metadata, such as section.
            for k, v in raw_record.items():
                if not isinstance(v, (dict, list)):
                    row.setdefault(k, v)

            rows.append(row)

    df = pd.DataFrame(rows)

    print(f"Flattened JSONL shape: {df.shape}")
    print("First 20 columns:")
    print(df.columns[:20].tolist())

    return df
for split_name in ["train", "valid", "test"]:
    jsonl_path = Path(DATASET_CONFIG[f"{split_name}_path"])
    parquet_path = jsonl_path.with_suffix(".parquet")

    if parquet_path.exists():
        parquet_path.unlink()
        print(f"Deleted old parquet cache: {parquet_path}")
def load_table(path, use_parquet_cache=True, save_parquet_cache=True):
    """
    Load a JSONL dataset with optional parquet caching.

    Expected workflow:
    - Input path is a .jsonl file.
    - If same-name .parquet already exists, load the parquet file.
    - If same-name .parquet does not exist, load and flatten the JSONL file.
    - Then save the flattened dataframe as .parquet for faster future loading.

    Example:
    Input:
        universe_100_recent_post_normalisation_train.jsonl

    Cache:
        universe_100_recent_post_normalisation_train.parquet
    """
    path = Path(path)

    if path.suffix.lower() != ".jsonl":
        raise ValueError(
            f"This loader expects a .jsonl input path, but got: {path}"
        )

    parquet_path = path.with_suffix(".parquet")

    if use_parquet_cache and parquet_path.exists():
        print("Parquet cache found. Loading parquet instead of JSONL:")
        print(f"  {parquet_path}")
        df = pd.read_parquet(parquet_path)

    else:
        if not path.exists():
            raise FileNotFoundError(f"JSONL file not found: {path}")

        print("No parquet cache found. Loading and flattening JSONL:")
        print(f"  {path}")

        df = load_flatten_jsonl(path)

        if save_parquet_cache:
            print("Saving flattened JSONL to parquet cache:")
            print(f"  {parquet_path}")
            parquet_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_parquet(parquet_path, index=False)

    print(f"Loaded dataframe shape: {df.shape}")

    return df



def standardise_frame(df, split_name):
    df = df.copy()
    df[SPLIT_COL] = split_name
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
        df[YEAR_COL] = df[DATE_COL].dt.year
    for col in [TICKER_COL, SIC2_COL]:
        if col not in df.columns:
            df[col] = np.nan
    return df


def load_dataset_from_config(config):
    frames = {}
    for split_name in ['train', 'valid', 'test']:
        df = load_table(config[f'{split_name}_path'])
        frames[split_name] = standardise_frame(df, split_name)
        print(f'{split_name}: {frames[split_name].shape}')
    all_data = pd.concat(frames.values(), ignore_index=True, sort=False)
    return frames, all_data

frames, all_data = load_dataset_from_config(DATASET_CONFIG)

Deleted old parquet cache: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_train.parquet
Deleted old parquet cache: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_validation.parquet
Deleted old parquet cache: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_test.parquet
No parquet cache found. Loading and flattening JSONL:
  c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_train.jsonl
Flattened JSONL shape: (98419, 748)
First 20 columns:
['IndexReference', 'attr__b_asset_turnover_better_is_higher', 'attr__b_assets_to_equity_better_is_higher', 'attr__b_cash_conversion_cycle_better_is_higher', 'attr__b_cfo_margin_better_is_higher', 'attr__b_current_assets_to_total_liabilities_better_is_higher', 'attr__b_current_ratio_better_is_higher

## 4. Export manager

This section creates a central export system. Every EDA table, metric table, comparison table, diagnostic table, and checklist can be registered once and then exported automatically. This ensures every important DataFrame is available as Excel output.

In [49]:
EXPORTED_TABLES = {}


def safe_table_name(name, max_len=120):
    name = re.sub(r'[^A-Za-z0-9_\-]+', '_', str(name))
    name = re.sub(r'_+', '_', name).strip('_')
    return name[:max_len]


def safe_sheet_name(name):
    name = re.sub(r'[\[\]\:\*\?\/\\]', '_', str(name))[:31]
    return name or 'Sheet'


def register_table(name, df, export_immediately=False):
    if df is None:
        return None
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)
    key = safe_table_name(name)
    EXPORTED_TABLES[key] = df.copy()
    if export_immediately:
        path = TABLE_DIR / f'{key}.xlsx'
        df.to_excel(path, index=False)
        print('Exported:', path)
    return df

def make_excel_safe(df):
    """
    Convert dataframe values into Excel-safe formats.

    Main issue:
    Excel does not support timezone-aware datetimes.
    This function removes timezone information before writing to Excel.

    This only affects Excel export.
    It does not change the original dataframe used for modelling,
    parquet export, or JSON prediction export.
    """
    df = df.copy()

    for col in df.columns:
        # Case 1: pandas datetime64 with timezone
        if pd.api.types.is_datetime64tz_dtype(df[col]):
            df[col] = df[col].dt.tz_convert(None)

        # Case 2: object columns that may contain Timestamp objects with timezone
        elif df[col].dtype == "object":
            df[col] = df[col].apply(
                lambda x: x.tz_convert(None)
                if isinstance(x, pd.Timestamp) and x.tzinfo is not None
                else x
            )

    return df

def export_registered_tables(workbook_name=None, export_individual_files=True, max_sheet_rows=1_000_000):
    if workbook_name is None:
        workbook_name = f'integrated_experiment_tables_{RUN_TIMESTAMP}.xlsx'
    workbook_path = TABLE_DIR / workbook_name
    used = set()
    with pd.ExcelWriter(workbook_path, engine='openpyxl') as writer:
        for name, df in EXPORTED_TABLES.items():
            sheet = safe_sheet_name(name)
            base = sheet
            i = 1
            while sheet in used:
                suffix = f'_{i}'
                sheet = safe_sheet_name(base[:31-len(suffix)] + suffix)
                i += 1
            used.add(sheet)
            excel_df = make_excel_safe(df)
            excel_df.head(max_sheet_rows).to_excel(writer, sheet_name=sheet, index=False)
            
    if export_individual_files:
        for name, df in EXPORTED_TABLES.items():
            excel_df = make_excel_safe(df)
            excel_df.head(max_sheet_rows).to_excel(TABLE_DIR / f'{name}.xlsx', index=False)
    print('Integrated workbook exported to:', workbook_path.resolve())
    print('Number of registered tables:', len(EXPORTED_TABLES))
    return workbook_path

## 5. Label construction and target configuration

This section standardises all target definitions. It preserves the EDA for each label and makes target handling explicit, so later modelling functions do not rely on hidden variable names.

Supported target families include perfect-hindsight labels, RL expert action labels, RL long-side quality/reward labels, and current PnL labels.

In [ ]:
def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def make_pi_hindsight_entry_long_6bins(series):
    """
    Convert pi_hindsight_entry_long into 6 semantic range-based bins.

    Interpretation:
    Class 0: y == 0
        No hindsight long-entry signal.

    Class 1: 0 < y < 0.1
        Very weak near-miss signal.
        Often consistent with base near-miss weight 0.1 plus price-distance penalty.

    Class 2: 0.1 <= y < 0.2
        Weak near-miss signal.

    Class 3: 0.2 <= y < 0.4
        Stronger near-miss / near-entry zone.
        Near-miss score can approach ~0.38 if base weight 0.3 receives price-distance boost.

    Class 4: 0.4 <= y < 0.7
        Weak-to-moderate original hindsight entry signal.

    Class 5: 0.7 <= y <= 1.0
        Strong original hindsight entry signal.


    """
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=s.index)

    out[s == 0] = 0
    out[(s > 0) & (s < 0.1)] = 1
    out[(s >= 0.1) & (s < 0.2)] = 2
    out[(s >= 0.2) & (s < 0.4)] = 3
    out[(s >= 0.4) & (s < 0.7)] = 4
    out[(s >= 0.7) & (s <= 1.0)] = 5

    return out.astype("Int64")


def add_derived_targets(df):
    """
    Construct modelling targets from the original label columns.

    The original Adaptive Swarm labels keep the label__ prefix.
    This function creates target__ columns that are used for the experiment's clarity
    """
    df = df.copy()

    # ------------------------------------------------------------
    # Perfect hindsight long-entry targets
    # ------------------------------------------------------------
    df["target__pi_hindsight_entry_long"] = pd.to_numeric(df["label__pi_hindsight_entry_long"], errors="coerce")
    df["target__pi_hindsight_entry_positive"] = (df["target__pi_hindsight_entry_long"] > 0).astype("Int64")
    df["target__pi_hindsight_entry_original"] = (df["target__pi_hindsight_entry_long"] >= 0.4).astype("Int64")
    df["target__pi_hindsight_entry_6bins"] = make_pi_hindsight_entry_long_6bins(df["target__pi_hindsight_entry_long"])

    # ------------------------------------------------------------
    # RL evaluator targets
    # ------------------------------------------------------------
    df["target__rl_expert_action"] = df["label__rl.expert_action"] # 3 class
    df["target__rl_long_action_label"] = df["label__rl.long.action_label"] # binary
    df["target__rl_long_action_quality"] = pd.to_numeric(df["label__rl.long.action_quality"], errors="coerce") # continuous
    df["target__rl_long_reward"] = pd.to_numeric(df["label__rl.reward.long"], errors="coerce") # continuous
    df["target__rl_long_current_pnl"] = pd.to_numeric(df["label__rl.long.current_pnl"], errors="coerce") # continuous

    # ------------------------------------------------------------
    # Derived binary target: whether long is the best action
    # ------------------------------------------------------------
    r_long = pd.to_numeric(df["label__rl.reward.long"], errors="coerce")
    r_short = pd.to_numeric(df["label__rl.reward.short"], errors="coerce")
    r_no_trade = pd.to_numeric(df["label__rl.reward.no_trade"], errors="coerce")

    df["target__rl_long_is_best"] = ((r_long > r_short) & (r_long > r_no_trade)).astype("Int64")
    
    # Derived continuous target: Reward margin: long reward minus the best non-long reward
    best_non_long_reward = pd.concat([r_short, r_no_trade], axis=1).max(axis=1)
    df["target__rl_long_reward_margin"] = r_long - best_non_long_reward

    # Derived Binary target: High-confidence long:
    # long is better than both alternatives by at least this margin.
    HIGH_CONFIDENCE_MARGIN = 0.05
    df["target__rl_long_high_confidence"] = (
    df["target__rl_long_reward_margin"] >= HIGH_CONFIDENCE_MARGIN).astype("Int64")

    return df


TARGET_CONFIGS = {
    "pi_hindsight_entry_long": {"column": "target__pi_hindsight_entry_long", "task": "regression", "direction": "higher_is_better", "description": "Continuous perfect-hindsight long-entry score."},
    "pi_hindsight_entry_positive": {"column": "target__pi_hindsight_entry_positive", "task": "binary", "positive_label": 1, "direction": "higher_is_better", "description": "Binary target: pi hindsight long-entry score > 0."},
    "pi_hindsight_entry_original": {"column": "target__pi_hindsight_entry_original", "task": "binary", "positive_label": 1, "direction": "higher_is_better", "description": "Binary target: pi hindsight long-entry score >= 0.4, representing original hindsight entry signals."},
    "pi_hindsight_entry_6bins": {"column": "target__pi_hindsight_entry_6bins", "task": "multiclass", "direction": "higher_is_better", "description": "Six-bin ordinal perfect-hindsight long-entry target."},
    "rl_expert_action": {"column": "target__rl_expert_action", "task": "multiclass", "direction": "action", "description": "RL evaluator expert action classification target."},
    "rl_long_action_quality": {"column": "target__rl_long_action_quality", "task": "regression", "direction": "higher_is_better", "description": "Continuous RL long action quality score."},
    "rl_long_reward": {"column": "target__rl_long_reward", "task": "regression", "direction": "higher_is_better", "description": "Continuous RL reward for choosing the long action."},
    "rl_long_current_pnl": {"column": "target__rl_long_current_pnl", "task": "regression", "direction": "higher_is_better", "description": "Continuous current PnL associated with the long action."},
    "rl_long_is_best": {"column": "target__rl_long_is_best", "task": "binary", "positive_label": 1, "direction": "higher_is_better", "description": "Binary target: long reward is higher than both short reward and no-trade reward."},
    "rl_long_high_confidence": {"column": "target__rl_long_high_confidence", "task": "binary", "positive_label": 1, "direction": "higher_is_better", "description": "Binary target: long reward exceeds the best non-long alternative by at least the high-confidence margin."},
    "rl_long_reward_margin": {"column": "target__rl_long_reward_margin", "task": "regression", "direction": "higher_is_better", "description": "Continuous reward margin: long reward minus the best non-long reward."}
}

def apply_target_construction(frames):
    updated = {split: add_derived_targets(df) for split, df in frames.items()}
    all_data = pd.concat(updated.values(), ignore_index=True, sort=False)
    return updated, all_data

frames, all_data = apply_target_construction(frames)

In [ ]:
def get_column_inventory(df, prefix):
    """
    Return sorted columns that start with a given prefix.
    """
    return sorted([col for col in df.columns if col.startswith(prefix)])


def get_other_columns(df):
    """
    Return columns that are not attribute, feature, label, or target columns.
    """
    known_prefixes = ("attr__", "feature__", "label__", "target__")
    return sorted([col for col in df.columns if not col.startswith(known_prefixes)])


def build_column_inventory_tables(frames, all_data, output_dir=OUTPUT_DIR):
    """
    Build and export column inventories for:
    - attribute columns
    - feature columns
    - label columns
    - target columns
    - other columns

    The inventory is based on:
    - train split
    - validation split
    - test split
    - combined all_data
    """
    output_dir = Path(output_dir)
    inventory_dir = output_dir / "column_inventories"
    inventory_dir.mkdir(parents=True, exist_ok=True)

    inventory_rows = []
    inventory_lists = {}

    datasets = {"train": frames.get("train"), "valid": frames.get("valid"), "test": frames.get("test"), "all_data": all_data}

    prefix_map = {"attribute": "attr__", "feature": "feature__", "label": "label__", "target": "target__"}

    for dataset_name, df in datasets.items():
        if df is None:
            continue

        for column_type, prefix in prefix_map.items():
            cols = get_column_inventory(df, prefix)

            inventory_lists[f"{dataset_name}_{column_type}_cols"] = cols

            inventory_rows.append({
                "dataset": dataset_name,
                "column_type": column_type,
                "prefix": prefix,
                "n_columns": len(cols),
            })

        other_cols = get_other_columns(df)

        inventory_lists[f"{dataset_name}_other_cols"] = other_cols

        inventory_rows.append({
            "dataset": dataset_name,
            "column_type": "other",
            "prefix": "not attr__/feature__/label__/target__",
            "n_columns": len(other_cols),
        })

    inventory_summary = pd.DataFrame(inventory_rows)

    # ------------------------------------------------------------
    # Build long-format detailed inventory table
    # ------------------------------------------------------------
    detail_rows = []

    for list_name, cols in inventory_lists.items():
        parts = list_name.split("_")
        dataset_name = parts[0]
        column_type = parts[1]

        for col in cols:
            detail_rows.append({
                "inventory_name": list_name,
                "dataset": dataset_name,
                "column_type": column_type,
                "column": col,
            })

    inventory_detail = pd.DataFrame(detail_rows)

    # ------------------------------------------------------------
    # Save TXT inventories
    # ------------------------------------------------------------
    for list_name, cols in inventory_lists.items():
        txt_path = inventory_dir / f"{list_name}.txt"

        with txt_path.open("w", encoding="utf-8") as f:
            for col in cols:
                f.write(col + "\n")

    # ------------------------------------------------------------
    # Register Excel tables
    # ------------------------------------------------------------
    register_table("column_inventory_summary", inventory_summary)
    register_table("column_inventory_detail", inventory_detail)

    # ------------------------------------------------------------
    # Save one Excel workbook specifically for inventories
    # ------------------------------------------------------------
    inventory_excel_path = inventory_dir / "column_inventory.xlsx"

    with pd.ExcelWriter(inventory_excel_path, engine="openpyxl") as writer:
        inventory_summary.to_excel(writer, sheet_name="summary", index=False)
        inventory_detail.to_excel(writer, sheet_name="detail", index=False)

        for list_name, cols in inventory_lists.items():
            sheet_df = pd.DataFrame({"column": cols})
            sheet_name = list_name[:31]
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print("Column inventory summary:")
    display(inventory_summary)

    print("\nColumn inventory files saved to:")
    print(inventory_dir)

    print("\nColumn inventory Excel workbook saved to:")
    print(inventory_excel_path)

    return {
        "summary": inventory_summary,
        "detail": inventory_detail,
        "lists": inventory_lists,
        "inventory_dir": inventory_dir,
        "inventory_excel_path": inventory_excel_path,
    }
    # column_inventory = build_column_inventory_tables(frames, all_data, OUTPUT_DIR)

## 6. Label EDA

This section preserves EDA for each label before model training. This is important because the modelling results can only be interpreted correctly after checking label sparsity, split stability, time variation, ticker heterogeneity, and sector heterogeneity.

In [ ]:
def available_target_configs(df, target_configs=TARGET_CONFIGS):
    return {name: cfg for name, cfg in target_configs.items() if cfg['column'] in df.columns}


def label_overall_summary(all_data, target_configs=TARGET_CONFIGS):
    rows = []
    for target_name, cfg in available_target_configs(all_data, target_configs).items():
        col = cfg['column']
        y = all_data[col]
        row = {'target_name': target_name, 'column': col, 'task': cfg['task'], 'description': cfg.get('description', ''), 'n_rows': len(y), 'n_non_missing': int(y.notna().sum()), 'missing_rate': y.isna().mean(), 'n_unique': y.nunique(dropna=True)}
        if cfg['task'] == 'regression':
            yy = pd.to_numeric(y, errors='coerce')
            row.update({'mean': yy.mean(), 'std': yy.std(), 'min': yy.min(), 'p01': yy.quantile(0.01), 'p05': yy.quantile(0.05), 'median': yy.median(), 'p95': yy.quantile(0.95), 'p99': yy.quantile(0.99), 'max': yy.max(), 'zero_rate': (yy == 0).mean(), 'positive_rate': (yy > 0).mean(), 'negative_rate': (yy < 0).mean()})
        else:
            if cfg['task'] == 'binary':
                row['positive_rate'] = (y == cfg.get('positive_label', 1)).mean()
            counts = y.value_counts(dropna=False, normalize=True)
            for k, v in counts.head(20).items():
                row[f'class_rate_{k}'] = v
        rows.append(row)
    return register_table('label_eda_overall_summary', pd.DataFrame(rows))


def label_distribution_by_split(all_data, target_configs=TARGET_CONFIGS):
    rows = []
    for target_name, cfg in available_target_configs(all_data, target_configs).items():
        col = cfg['column']
        for split_name, g in all_data.groupby(SPLIT_COL, dropna=False):
            y = g[col]
            base = {'target_name': target_name, 'split': split_name, 'task': cfg['task'], 'n_rows': len(g), 'n_non_missing': int(y.notna().sum()), 'missing_rate': y.isna().mean()}
            if cfg['task'] == 'regression':
                yy = pd.to_numeric(y, errors='coerce')
                base.update({'mean': yy.mean(), 'std': yy.std(), 'median': yy.median(), 'zero_rate': (yy == 0).mean(), 'positive_rate': (yy > 0).mean(), 'negative_rate': (yy < 0).mean()})
                rows.append(base)
            else:
                for klass, rate in y.value_counts(dropna=False, normalize=True).items():
                    row = base.copy(); row['class'] = klass; row['class_rate'] = rate; row['class_count'] = int((y == klass).sum()) if pd.notna(klass) else int(y.isna().sum()); rows.append(row)
    return register_table('label_eda_by_split', pd.DataFrame(rows))


def label_distribution_by_group(all_data, group_col, target_configs=TARGET_CONFIGS, min_rows=30):
    if group_col not in all_data.columns:
        return register_table(f'label_eda_by_{group_col}', pd.DataFrame())
    rows = []
    for target_name, cfg in available_target_configs(all_data, target_configs).items():
        col = cfg['column']
        for group_value, g in all_data.groupby(group_col, dropna=False):
            if len(g) < min_rows:
                continue
            y = g[col]
            row = {'target_name': target_name, 'group_col': group_col, 'group_value': group_value, 'task': cfg['task'], 'n_rows': len(g), 'n_non_missing': int(y.notna().sum()), 'missing_rate': y.isna().mean()}
            if cfg['task'] == 'regression':
                yy = pd.to_numeric(y, errors='coerce')
                row.update({'mean': yy.mean(), 'std': yy.std(), 'median': yy.median(), 'zero_rate': (yy == 0).mean(), 'positive_rate': (yy > 0).mean(), 'negative_rate': (yy < 0).mean()})
            else:
                if cfg['task'] == 'binary':
                    row['positive_rate'] = (y == cfg.get('positive_label', 1)).mean()
                row['mode'] = y.mode(dropna=True).iloc[0] if y.notna().any() else np.nan
                row['mode_rate'] = y.value_counts(normalize=True, dropna=True).iloc[0] if y.notna().any() else np.nan
            rows.append(row)
    return register_table(f'label_eda_by_{safe_table_name(group_col)}', pd.DataFrame(rows))


def run_all_label_eda(all_data):
    tables = {
        "overall": label_overall_summary(all_data),
        "by_split": label_distribution_by_split(all_data),
        "by_year": label_distribution_by_group(all_data, YEAR_COL),
        "by_ticker": label_distribution_by_group(all_data, TICKER_COL),
        "by_sic2": label_distribution_by_group(all_data, SIC2_COL),
    }

    register_table("label_eda_by_year", tables["by_year"])
    register_table("label_eda_by_ticker", tables["by_ticker"])
    register_table("label_eda_by_sic2", tables["by_sic2"])

    return tables

# ============================================================
# RL trainable flag EDA
# ============================================================
# Purpose:
# Check how many observations are marked as RL-trainable.
#
# Why this matters:
# For RL-derived targets, observations where label__rl.trainable == 0
# should usually be excluded from both model training and evaluation.
# This EDA shows how much data remains after applying that filter.
# ============================================================

RL_TRAINABLE_COL = "label__rl.trainable"


def run_rl_trainable_eda(all_data):
    tables = {}

    if RL_TRAINABLE_COL not in all_data.columns:
        print(f"{RL_TRAINABLE_COL} not found. Skipping RL trainable EDA.")
        return tables

    trainable_by_split = (
        all_data
        .groupby(SPLIT_COL)[RL_TRAINABLE_COL]
        .agg(
            n="size",
            n_trainable="sum",
            trainable_rate="mean"
        )
        .reset_index()
    )

    tables["rl_trainable_by_split"] = register_table(
        "rl_trainable_by_split",
        trainable_by_split
    )

    if YEAR_COL in all_data.columns:
        trainable_by_year = (
            all_data
            .groupby([SPLIT_COL, YEAR_COL])[RL_TRAINABLE_COL]
            .agg(
                n="size",
                n_trainable="sum",
                trainable_rate="mean"
            )
            .reset_index()
        )

        tables["rl_trainable_by_year"] = register_table(
            "rl_trainable_by_year",
            trainable_by_year
        )

    if TICKER_COL in all_data.columns:
        trainable_by_ticker = (
            all_data
            .groupby([SPLIT_COL, TICKER_COL])[RL_TRAINABLE_COL]
            .agg(
                n="size",
                n_trainable="sum",
                trainable_rate="mean"
            )
            .reset_index()
        )

        tables["rl_trainable_by_ticker"] = register_table(
            "rl_trainable_by_ticker",
            trainable_by_ticker
        )

    return tables

rl_trainable_eda_tables = run_rl_trainable_eda(all_data)
label_eda_tables = run_all_label_eda(all_data)


## 7. Feature-set construction

In [ ]:
EXCLUDE_COLUMNS = {SPLIT_COL, YEAR_COL, DATE_COL, TICKER_COL, SIC2_COL}


def read_feature_file(feature_file):
    if feature_file is None:
        return []
    path = Path(feature_file)
    if not path.exists():
        print(f'Feature file not found: {path}. Falling back to automatic numeric features.')
        return []
    raw = [line.strip() for line in path.open('r', encoding='utf-8') if line.strip() and not line.strip().startswith('#')]
    return [col if col.startswith('feature__') else f'feature__{col}' for col in raw]


def get_numeric_feature_candidates(df):
    numeric_cols = df.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    out = []
    for col in numeric_cols:
        if col in EXCLUDE_COLUMNS:
            continue
        if col.startswith('label__') or col.startswith('target__') or col.startswith('attr__'):
            continue
        if col.startswith('feature__'):
            out.append(col)
    return sorted(set(out))


def select_features_by_keywords(all_features, include_keywords=None, exclude_keywords=None):
    include_keywords = include_keywords or []
    exclude_keywords = exclude_keywords or []
    out = []
    for col in all_features:
        c = col.lower()
        include_ok = True if not include_keywords else any(k.lower() in c for k in include_keywords)
        exclude_ok = not any(k.lower() in c for k in exclude_keywords)
        if include_ok and exclude_ok:
            out.append(col)
    return sorted(set(out))


def build_feature_sets(all_data, config=DATASET_CONFIG):
    auto_numeric = get_numeric_feature_candidates(all_data)
    project_b_raw = read_feature_file(config.get('project_b_feature_file'))
    project_b = [c for c in project_b_raw if c in all_data.columns]
    if len(project_b) == 0:
        project_b = auto_numeric.copy()

    feature_sets = {
        'combined_project_b': project_b,
        'combined_all_numeric_features': auto_numeric,
        'fundamentals_only': select_features_by_keywords(auto_numeric, ['fund', 'asset', 'liabil', 'equity', 'cash', 'debt', 'revenue', 'income', 'earn', 'profit', 'margin', 'eps', 'book', 'balance', 'report', 'quarter', 'ttm', 'filing']),
        'daily_valuation_only': select_features_by_keywords(auto_numeric, ['pe', 'pb', 'ps', 'ev', 'valuation', 'market_cap', 'price_to', 'yield', 'dividend', 'multiple']),
        'momentum_volatility_only': select_features_by_keywords(auto_numeric, ['return', 'ret', 'momentum', 'mom', 'vol', 'volatility', 'atr', 'rsi', 'macd', 'sma', 'ema', 'drawdown', 'trend', 'beta']),
        'macro_regime_only': select_features_by_keywords(auto_numeric, ['macro', 'regime', 'inflation', 'vix', 'yield_curve', 'rate', 'treasury', 'credit', 'index', 'sector', 'market']),
        'calc_algo_only': select_features_by_keywords(auto_numeric, ['calc_algo']),
        'algorithmic_signals_only': select_features_by_keywords(auto_numeric, ['signal', 'alpha', 'score', 'rank', 'swarm', 'model', 'entry', 'exit', 'confidence'], ['label', 'target']),
    }
    feature_sets = {k: v for k, v in feature_sets.items() if len(v) > 0}
    summary = pd.DataFrame([{'feature_set': k, 'n_features': len(v)} for k, v in feature_sets.items()])
    register_table('feature_set_summary', summary)
    return feature_sets

FEATURE_SET_CONFIGS = build_feature_sets(all_data, DATASET_CONFIG)

## 8. Shared modelling utilities

This section contains functions that can be shared across regression, binary classification, and multiclass classification. Task-specific model dictionaries are separated where the analysis differs.

In [ ]:
def make_numeric_preprocessor(scale=False):
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    return Pipeline(steps)


def make_regression_models():
    models = {
        'DummyMean': DummyRegressor(strategy='mean'),
        'Ridge': make_pipeline(make_numeric_preprocessor(True), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        'Lasso': make_pipeline(make_numeric_preprocessor(True), Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=5000)),
        'ElasticNet': make_pipeline(make_numeric_preprocessor(True), ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMRegressor
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def make_binary_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'DummyStratified': DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
        'Logistic': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def make_multiclass_models():
    models = {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
        'LogisticMultinomial': make_pipeline(make_numeric_preprocessor(True), LogisticRegression(class_weight='balanced', max_iter=3000, multi_class='auto', random_state=RANDOM_STATE)),
        'RandomForest': make_pipeline(make_numeric_preprocessor(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS)),
        'HistGradientBoosting': make_pipeline(make_numeric_preprocessor(False), HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = make_pipeline(make_numeric_preprocessor(False), LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1))
    except Exception:
        pass
    return models


def get_models_for_task(task):
    if task == 'regression': return make_regression_models()
    if task == 'binary': return make_binary_models()
    if task == 'multiclass': return make_multiclass_models()
    raise ValueError(f'Unknown task: {task}')

def is_rl_target(target_name):
    """
    Return True if the target is derived from RL evaluator labels.
    """
    return target_name.startswith("rl_")

RL_TRAINABLE_COL = "label__rl.trainable"
def filter_rows_for_target(df, target_name):
    """
    Filter rows according to the target type.

    For RL targets:
    - If label__rl.trainable exists, keep only rows with rl.trainable == 1.

    For non-RL targets:
    - Keep all rows.
    """
    df = df.copy()

    if is_rl_target(target_name) and RL_TRAINABLE_COL in df.columns:
        before = len(df)
        df = df[df[RL_TRAINABLE_COL] == 1].copy()
        after = len(df)

        # Store numbers for debugging if needed.
        # print(f"{target_name}: filtered rl.trainable rows {before} -> {after}")

    return df

def prepare_xy(df, feature_cols, target_col, task):
    cols = [c for c in feature_cols if c in df.columns] + [target_col]
    data = df[cols].copy().dropna(subset=[target_col])
    X = data[[c for c in feature_cols if c in data.columns]]
    y = data[target_col]
    if task == 'regression':
        y = pd.to_numeric(y, errors='coerce')
        valid = y.notna(); X = X.loc[valid]; y = y.loc[valid]
    else:
        valid = y.notna(); X = X.loc[valid]; y = y.loc[valid]
    return X, y


def safe_spearman(y_true, y_score):
    y_true = pd.Series(y_true); y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='spearman')


def safe_pearson(y_true, y_score):
    y_true = pd.Series(y_true); y_score = pd.Series(y_score)
    valid = y_true.notna() & y_score.notna()
    if valid.sum() < 3 or y_true.loc[valid].nunique() < 2 or y_score.loc[valid].nunique() < 2:
        return np.nan
    return y_true.loc[valid].corr(y_score.loc[valid], method='pearson')


def get_positive_proba(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        classes = getattr(model, 'classes_', None)
        if classes is None and hasattr(model, 'named_steps'):
            classes = getattr(list(model.named_steps.values())[-1], 'classes_', None)
        if classes is not None:
            classes = list(classes)
            if 1 in classes: return proba[:, classes.index(1)]
            if True in classes: return proba[:, classes.index(True)]
        return proba[:, -1]
    if hasattr(model, 'decision_function'):
        score = model.decision_function(X)
        return 1 / (1 + np.exp(-score))
    return None


def build_prediction_frame(df, y_true, y_pred, score, target_name, task, feature_set_name, model_name, split_name):
    meta_cols = [DATE_COL, YEAR_COL, TICKER_COL, SIC2_COL]
    meta = df.loc[y_true.index, [c for c in meta_cols if c in df.columns]].copy()
    out = meta.copy()
    out['target_name'] = target_name; out['task'] = task; out['feature_set'] = feature_set_name; out['model'] = model_name; out[SPLIT_COL] = split_name
    out[TRUE_COL] = np.asarray(y_true); out[PRED_COL] = np.asarray(y_pred); out[SCORE_COL] = np.asarray(score) if score is not None else np.asarray(y_pred)
    return out

## 9. Metric functions

This section separates task-specific metrics from shared modelling logic. Regression, binary classification, multiclass classification, calibration, and ranking metrics answer different research questions. Ranking and Top-K metrics are especially important because the practical question is whether the model ranks better long opportunities above weaker ones.

In [37]:
def regression_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float); y_pred = pd.Series(y_pred).astype(float)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0: return {}
    yt = y_true.loc[valid]; yp = y_pred.loc[valid]
    return {'n': int(valid.sum()), 'mae': mean_absolute_error(yt, yp), 'rmse': np.sqrt(mean_squared_error(yt, yp)), 'r2': r2_score(yt, yp) if yt.nunique() > 1 else np.nan, 'pearson': safe_pearson(yt, yp), 'spearman': safe_spearman(yt, yp), 'mean_y_true': yt.mean(), 'mean_y_pred': yp.mean(), 'std_y_true': yt.std(), 'std_y_pred': yp.std()}


def binary_metrics(y_true, y_pred, y_score=None):
    y_true = pd.Series(y_true); y_pred = pd.Series(y_pred)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0: return {}
    yt = y_true.loc[valid].astype(int); yp = y_pred.loc[valid].astype(int)
    out = {'n': int(valid.sum()), 'accuracy': accuracy_score(yt, yp), 'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan, 'precision': precision_score(yt, yp, zero_division=0), 'recall': recall_score(yt, yp, zero_division=0), 'f1': f1_score(yt, yp, zero_division=0), 'positive_rate_true': yt.mean(), 'positive_rate_pred': yp.mean()}
    if y_score is not None:
        ys = pd.Series(y_score, index=yt.index).astype(float)
        out['mean_predicted_probability'] = ys.mean()
        if yt.nunique() > 1 and ys.nunique() > 1:
            out.update({'roc_auc': roc_auc_score(yt, ys), 'pr_auc': average_precision_score(yt, ys), 'brier_score': brier_score_loss(yt, np.clip(ys, 0, 1)), 'spearman': safe_spearman(yt, ys)})
    return out


def multiclass_metrics(y_true, y_pred, y_proba=None):
    y_true = pd.Series(y_true); y_pred = pd.Series(y_pred)
    valid = y_true.notna() & y_pred.notna()
    if valid.sum() == 0: return {}
    yt = y_true.loc[valid]; yp = y_pred.loc[valid]
    out = {'n': int(valid.sum()), 'accuracy': accuracy_score(yt, yp), 'balanced_accuracy': balanced_accuracy_score(yt, yp) if yt.nunique() > 1 else np.nan, 'macro_f1': f1_score(yt, yp, average='macro', zero_division=0), 'weighted_f1': f1_score(yt, yp, average='weighted', zero_division=0), 'n_classes_true': yt.nunique(), 'n_classes_pred': yp.nunique()}
    try: out['spearman_class_rank'] = safe_spearman(pd.to_numeric(yt), pd.to_numeric(yp))
    except Exception: out['spearman_class_rank'] = np.nan
    return out


def calibration_table(y_true, y_score, n_bins=10):
    df = pd.DataFrame({TRUE_COL: pd.Series(y_true).astype(float), SCORE_COL: pd.Series(y_score).astype(float)}).dropna()
    if len(df) == 0: return pd.DataFrame()
    df[SCORE_COL] = df[SCORE_COL].clip(0, 1)
    df['prob_bin'] = pd.cut(df[SCORE_COL], bins=np.linspace(0, 1, n_bins + 1), include_lowest=True)
    out = df.groupby('prob_bin', observed=False).agg(n=(TRUE_COL, 'size'), mean_predicted_probability=(SCORE_COL, 'mean'), true_positive_rate=(TRUE_COL, 'mean')).reset_index()
    out['calibration_error'] = out['mean_predicted_probability'] - out['true_positive_rate']
    return out


def daily_cross_sectional_spearman(pred_df, min_daily_rows=5):
    rows = []
    if DATE_COL not in pred_df.columns: return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        if len(g) < min_daily_rows: continue
        rows.append({DATE_COL: date, 'n': len(g), 'daily_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]), 'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(), 'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean()})
    return pd.DataFrame(rows)


def topk_diagnostics(pred_df, top_pct=0.10, min_daily_rows=10):
    rows = []
    if DATE_COL not in pred_df.columns: return pd.DataFrame()
    for date, g in pred_df.groupby(DATE_COL):
        g = g.dropna(subset=[TRUE_COL, SCORE_COL]).copy()
        if len(g) < min_daily_rows: continue
        k = max(1, int(math.ceil(len(g) * top_pct)))
        top = g.nlargest(k, SCORE_COL); bottom = g.nsmallest(k, SCORE_COL)
        y = pd.to_numeric(g[TRUE_COL], errors='coerce'); top_y = pd.to_numeric(top[TRUE_COL], errors='coerce'); bottom_y = pd.to_numeric(bottom[TRUE_COL], errors='coerce')
        row = {DATE_COL: date, 'n': len(g), 'k': k, 'top_pct': top_pct, 'overall_mean_true': y.mean(), 'top_mean_true': top_y.mean(), 'bottom_mean_true': bottom_y.mean(), 'top_minus_bottom_spread': top_y.mean() - bottom_y.mean()}
        unique_values = pd.Series(g[TRUE_COL]).dropna().unique()
        if set(unique_values).issubset({0, 1, False, True}):
            base_rate = y.mean(); precision_at_k = top_y.mean()
            row.update({'base_positive_rate': base_rate, 'precision_at_k': precision_at_k, 'bottom_positive_rate': bottom_y.mean(), 'lift_at_k': precision_at_k / base_rate if base_rate and base_rate > 0 else np.nan})
        rows.append(row)
    return pd.DataFrame(rows)


def within_ticker_ranking(pred_df, min_rows=20):
    if TICKER_COL not in pred_df.columns: return pd.DataFrame()
    rows = []
    for ticker, g in pred_df.groupby(TICKER_COL, dropna=False):
        if len(g) < min_rows: continue
        rows.append({'ticker': ticker, 'n': len(g), 'within_ticker_spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL]), 'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(), 'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean()})
    return pd.DataFrame(rows)


def subgroup_prediction_metrics(pred_df, group_col, min_rows=30):
    if group_col not in pred_df.columns: return pd.DataFrame()
    rows = []
    for group_value, g in pred_df.groupby(group_col, dropna=False):
        if len(g) < min_rows: continue
        row = {'group_col': group_col, 'group_value': group_value, 'n': len(g), 'mean_true': pd.to_numeric(g[TRUE_COL], errors='coerce').mean(), 'mean_score': pd.to_numeric(g[SCORE_COL], errors='coerce').mean(), 'spearman': safe_spearman(g[TRUE_COL], g[SCORE_COL])}
        unique_values = pd.Series(g[TRUE_COL]).dropna().unique()
        try:
            if set(unique_values).issubset({0, 1, False, True}): row.update(binary_metrics(g[TRUE_COL], g[PRED_COL], g[SCORE_COL]))
            else: row.update(regression_metrics(g[TRUE_COL], g[SCORE_COL]))
        except Exception:
            pass
        rows.append(row)
    return pd.DataFrame(rows)

## 10. Main experiment runner

This section is the core integrated runner. It loops over target definitions, feature sets, and models while keeping the same variable names and output format. The runner automatically skips invalid combinations, such as unavailable target columns, empty feature sets, or targets with too few training examples.

In [ ]:
EXPERIMENT_CONFIG = {
    'target_names': None,
    'feature_set_names': None,
    'model_names': None,
    'min_train_rows': 100,
    'min_valid_or_test_rows': 50,
    'save_row_level_predictions': True,
    'run_calibration': True,
    'run_daily_ranking': True,
    'run_topk': True,
    'run_within_ticker_ranking': True,
    'run_subgroup_diagnostics': True,
    'topk_percentages': [0.05, 0.10],
}


def should_run_name(name, selected_names):
    return selected_names is None or name in selected_names


def fit_predict_single_experiment(frames, target_name, target_cfg, feature_set_name, feature_cols, model_name, model, config=EXPERIMENT_CONFIG):
    target_col = target_cfg['column']; task = target_cfg['task']
    X_train, y_train = prepare_xy(frames['train'], feature_cols, target_col, task)
    if len(y_train) < config['min_train_rows']:
        return [], [], {'status': 'skipped', 'reason': 'too_few_train_rows', 'n_train': len(y_train)}
    if task in ['binary', 'multiclass'] and y_train.nunique(dropna=True) < 2:
        return [], [], {'status': 'skipped', 'reason': 'single_class_train', 'n_train': len(y_train)}
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)
    metric_rows = []; prediction_frames = []
    for split_name in ['valid', 'test']:
        eval_df = frames[split_name]
        X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col, task)
        if len(y_eval) < config['min_valid_or_test_rows']:
            continue
        y_pred = fitted_model.predict(X_eval)
        if task == 'binary':
            y_score = get_positive_proba(fitted_model, X_eval)
            if y_score is None: y_score = y_pred
            metrics = binary_metrics(y_eval, y_pred, y_score)
        elif task == 'regression':
            y_score = y_pred; metrics = regression_metrics(y_eval, y_pred)
        elif task == 'multiclass':
            y_score = y_pred; metrics = multiclass_metrics(y_eval, y_pred)
        row = {'dataset_name': DATASET_CONFIG.get('dataset_name', ''), 'target_name': target_name, 'target_column': target_col, 'task': task, 'feature_set': feature_set_name, 'n_features': len(feature_cols), 'model': model_name, 'split': split_name}
        row.update(metrics); metric_rows.append(row)
        prediction_frames.append(build_prediction_frame(eval_df, y_eval, y_pred, y_score, target_name, task, feature_set_name, model_name, split_name))
    return metric_rows, prediction_frames, {'status': 'completed', 'n_train': len(y_train)}

def fit_predict_single_experiment(frames, target_name, target_cfg, feature_set_name, feature_cols, model_name, model, config=EXPERIMENT_CONFIG):
    """
    Fit one model for one target and one feature set.

    Important:
    For RL-derived targets, this function applies the rl.trainable filter
    before preparing X/y. This means observations with label__rl.trainable == 0
    are excluded from train, validation, and test evaluation for RL targets.
    PI hindsight targets are not affected by this RL-specific filter.
    """
    target_col = target_cfg["column"]
    task = target_cfg["task"]

    # ------------------------------------------------------------
    # Apply target-specific row filtering
    # ------------------------------------------------------------
    train_df = filter_rows_for_target(frames["train"], target_name)
    valid_df = filter_rows_for_target(frames["valid"], target_name)
    test_df = filter_rows_for_target(frames["test"], target_name)

    # ------------------------------------------------------------
    # Prepare training data
    # ------------------------------------------------------------
    X_train, y_train = prepare_xy(train_df, feature_cols, target_col, task)

    if len(y_train) < config["min_train_rows"]:
        return [], [], {
            "status": "skipped",
            "reason": "too_few_train_rows_after_target_filter",
            "n_train": len(y_train)
        }

    if task in ["binary", "multiclass"] and y_train.nunique(dropna=True) < 2:
        return [], [], {
            "status": "skipped",
            "reason": "single_class_train_after_target_filter",
            "n_train": len(y_train)
        }

    # ------------------------------------------------------------
    # Fit model
    # ------------------------------------------------------------
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    metric_rows = []
    prediction_frames = []

    # ------------------------------------------------------------
    # Evaluate on validation and test
    # ------------------------------------------------------------
    for split_name, eval_df in [("valid", valid_df), ("test", test_df)]:
        X_eval, y_eval = prepare_xy(eval_df, feature_cols, target_col, task)

        if len(y_eval) < config["min_valid_or_test_rows"]:
            continue

        y_pred = fitted_model.predict(X_eval)

        if task == "binary":
            y_score = get_positive_proba(fitted_model, X_eval)
            if y_score is None:
                y_score = y_pred
            metrics = binary_metrics(y_eval, y_pred, y_score)

        elif task == "regression":
            y_score = y_pred
            metrics = regression_metrics(y_eval, y_pred)

        elif task == "multiclass":
            y_score = y_pred
            metrics = multiclass_metrics(y_eval, y_pred)

        else:
            raise ValueError(f"Unknown task: {task}")

        row = {
            "dataset_name": DATASET_CONFIG.get("dataset_name", ""),
            "target_name": target_name,
            "target_column": target_col,
            "task": task,
            "feature_set": feature_set_name,
            "n_features": len(feature_cols),
            "model": model_name,
            "split": split_name,
            "n_train_after_target_filter": len(y_train),
            "n_eval_after_target_filter": len(y_eval),
        }

        row.update(metrics)
        metric_rows.append(row)

        prediction_frames.append(build_prediction_frame(eval_df, y_eval, y_pred, y_score, target_name, task, feature_set_name, model_name, split_name))
   

    return metric_rows, prediction_frames, {
        "status": "completed",
        "n_train": len(y_train)
    }
def run_integrated_experiments(frames, feature_sets, target_configs=TARGET_CONFIGS, config=EXPERIMENT_CONFIG):
    all_metric_rows = []; all_prediction_frames = []; skipped_rows = []
    all_tmp = pd.concat(frames.values(), ignore_index=True, sort=False)
    available_targets = available_target_configs(all_tmp, target_configs)
    for target_name, target_cfg in available_targets.items():
        if not should_run_name(target_name, config['target_names']): continue
        models = get_models_for_task(target_cfg['task'])
        for feature_set_name, feature_cols in feature_sets.items():
            if not should_run_name(feature_set_name, config['feature_set_names']): continue
            if len(feature_cols) == 0: continue
            for model_name, model in models.items():
                if not should_run_name(model_name, config['model_names']): continue
                print(f'Running: target={target_name} | features={feature_set_name} | model={model_name}')
                try:
                    metric_rows, prediction_frames, status = fit_predict_single_experiment(frames, target_name, target_cfg, feature_set_name, feature_cols, model_name, model, config)
                    all_metric_rows.extend(metric_rows); all_prediction_frames.extend(prediction_frames)
                    if status['status'] != 'completed':
                        skipped_rows.append({'target_name': target_name, 'feature_set': feature_set_name, 'model': model_name, **status})
                except Exception as e:
                    skipped_rows.append({'target_name': target_name, 'feature_set': feature_set_name, 'model': model_name, 'status': 'error', 'reason': str(e)})
                    print('  -> skipped/error:', e)
    metrics_df = pd.DataFrame(all_metric_rows)
    predictions_df = pd.concat(all_prediction_frames, ignore_index=True, sort=False) if all_prediction_frames else pd.DataFrame()
    skipped_df = pd.DataFrame(skipped_rows)
    register_table('model_metric_summary', metrics_df)
    register_table('skipped_or_failed_experiments', skipped_df)
    if config['save_row_level_predictions'] and len(predictions_df) > 0:
        pred_path = PREDICTION_DIR / f'integrated_predictions_{RUN_TIMESTAMP}.parquet'
        predictions_df.to_parquet(pred_path, index=False)
        print('Prediction parquet exported:', pred_path)
    return metrics_df, predictions_df, skipped_df

#metrics_df, predictions_df, skipped_df = run_integrated_experiments(frames, FEATURE_SET_CONFIGS)

## 11. Post-model diagnostics

This section turns row-level predictions into research-friendly diagnostic tables: daily cross-sectional Spearman, Top-K diagnostics, within-ticker ranking, subgroup diagnostics, and binary calibration tables. These outputs are essential for interpreting whether weak average performance hides useful subgroup or ranking behaviour.

In [47]:
def run_prediction_diagnostics(predictions_df, config=EXPERIMENT_CONFIG):
    if predictions_df is None or len(predictions_df) == 0:
        print('No predictions available.'); return {}
    diagnostic_tables = {}
    group_cols = ['target_name', 'task', 'feature_set', 'model', SPLIT_COL]

    if config['run_daily_ranking']:
        parts = []
        for keys, g in predictions_df.groupby(group_cols, dropna=False):
            one = daily_cross_sectional_spearman(g)
            if len(one) == 0: continue
            for col, value in zip(group_cols, keys): one[col] = value
            parts.append(one)
        daily_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        daily_summary = daily_df.groupby(group_cols, dropna=False).agg(n_days=('daily_spearman', 'count'), mean_daily_spearman=('daily_spearman', 'mean'), median_daily_spearman=('daily_spearman', 'median'), positive_spearman_day_rate=('daily_spearman', lambda x: (x > 0).mean())).reset_index() if len(daily_df) else pd.DataFrame()
        diagnostic_tables['daily_spearman_detail'] = register_table('daily_spearman_detail', daily_df)
        diagnostic_tables['daily_spearman_summary'] = register_table('daily_spearman_summary', daily_summary)

    if config['run_topk']:
        parts = []
        for top_pct in config['topk_percentages']:
            for keys, g in predictions_df.groupby(group_cols, dropna=False):
                one = topk_diagnostics(g, top_pct=top_pct)
                if len(one) == 0: continue
                for col, value in zip(group_cols, keys): one[col] = value
                parts.append(one)
        topk_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        if len(topk_df):
            agg = {'n_days': ('top_minus_bottom_spread', 'count'), 'mean_top_true': ('top_mean_true', 'mean'), 'mean_bottom_true': ('bottom_mean_true', 'mean'), 'mean_top_minus_bottom_spread': ('top_minus_bottom_spread', 'mean'), 'positive_spread_day_rate': ('top_minus_bottom_spread', lambda x: (x > 0).mean())}
            if 'precision_at_k' in topk_df.columns: agg['mean_precision_at_k'] = ('precision_at_k', 'mean')
            if 'lift_at_k' in topk_df.columns: agg['mean_lift_at_k'] = ('lift_at_k', 'mean')
            topk_summary = topk_df.groupby(group_cols + ['top_pct'], dropna=False).agg(**agg).reset_index()
        else:
            topk_summary = pd.DataFrame()
        diagnostic_tables['topk_detail'] = register_table('topk_detail', topk_df)
        diagnostic_tables['topk_summary'] = register_table('topk_summary', topk_summary)

    if config['run_within_ticker_ranking']:
        parts = []
        for keys, g in predictions_df.groupby(group_cols, dropna=False):
            one = within_ticker_ranking(g)
            if len(one) == 0: continue
            for col, value in zip(group_cols, keys): one[col] = value
            parts.append(one)
        w_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
        w_summary = w_df.groupby(group_cols, dropna=False).agg(n_tickers=('ticker', 'count'), mean_within_ticker_spearman=('within_ticker_spearman', 'mean'), median_within_ticker_spearman=('within_ticker_spearman', 'median'), positive_ticker_spearman_rate=('within_ticker_spearman', lambda x: (x > 0).mean())).reset_index() if len(w_df) else pd.DataFrame()
        diagnostic_tables['within_ticker_detail'] = register_table('within_ticker_detail', w_df)
        diagnostic_tables['within_ticker_summary'] = register_table('within_ticker_summary', w_summary)

    if config['run_subgroup_diagnostics']:
        for subgroup_col in [YEAR_COL, TICKER_COL, SIC2_COL]:
            if subgroup_col not in predictions_df.columns: continue
            parts = []
            for keys, g in predictions_df.groupby(group_cols, dropna=False):
                one = subgroup_prediction_metrics(g, subgroup_col)
                if len(one) == 0: continue
                for col, value in zip(group_cols, keys): one[col] = value
                parts.append(one)
            diagnostic_tables[f'subgroup_{subgroup_col}'] = register_table(f'subgroup_diagnostics_by_{subgroup_col}', pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame())

    if config['run_calibration']:
        parts = []
        binary_pred = predictions_df[predictions_df['task'] == 'binary'].copy()
        for keys, g in binary_pred.groupby(group_cols, dropna=False):
            one = calibration_table(g[TRUE_COL], g[SCORE_COL])
            if len(one) == 0: continue
            for col, value in zip(group_cols, keys): one[col] = value
            parts.append(one)
        diagnostic_tables['calibration'] = register_table('binary_calibration_tables', pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame())
    return diagnostic_tables

#diagnostic_tables = run_prediction_diagnostics(predictions_df)

## 12. Best-model comparison tables

This section creates compact tables for interpretation and dissertation writing. It identifies best models by target, best feature families, best daily-Spearman models, and best Top-K models.

In [40]:
def build_best_model_tables(metrics_df):
    if metrics_df is None or len(metrics_df) == 0: return {}
    df = metrics_df.copy()
    def choose_main_metric(row):
        if row['task'] == 'regression': return row.get('spearman', np.nan)
        if row['task'] == 'binary': return row.get('pr_auc', row.get('roc_auc', np.nan))
        if row['task'] == 'multiclass': return row.get('macro_f1', np.nan)
        return np.nan
    df['main_metric'] = df.apply(choose_main_metric, axis=1)
    best_by_target = df.dropna(subset=['main_metric']).sort_values(['target_name', 'split', 'main_metric'], ascending=[True, True, False]).groupby(['target_name', 'split'], as_index=False).head(1).reset_index(drop=True)
    feature_family = df.dropna(subset=['main_metric']).groupby(['target_name', 'task', 'feature_set', 'split'], dropna=False).agg(best_main_metric=('main_metric', 'max'), mean_main_metric=('main_metric', 'mean'), n_models=('model', 'nunique')).reset_index().sort_values(['target_name', 'split', 'best_main_metric'], ascending=[True, True, False])
    return {'best_by_target': register_table('best_model_by_target', best_by_target), 'feature_family_comparison': register_table('feature_family_comparison', feature_family)}


def build_best_ranking_tables():
    tables = {}
    daily = EXPORTED_TABLES.get('daily_spearman_summary')
    if daily is not None and len(daily) > 0:
        best_daily = daily.dropna(subset=['mean_daily_spearman']).sort_values(['target_name', SPLIT_COL, 'mean_daily_spearman'], ascending=[True, True, False]).groupby(['target_name', SPLIT_COL], as_index=False).head(1).reset_index(drop=True)
        tables['best_daily_spearman'] = register_table('best_daily_spearman_models', best_daily)
    topk = EXPORTED_TABLES.get('topk_summary')
    if topk is not None and len(topk) > 0:
        score_col = 'mean_lift_at_k' if 'mean_lift_at_k' in topk.columns else 'mean_top_minus_bottom_spread'
        best_topk = topk.dropna(subset=[score_col]).sort_values(['target_name', SPLIT_COL, 'top_pct', score_col], ascending=[True, True, True, False]).groupby(['target_name', SPLIT_COL, 'top_pct'], as_index=False).head(1).reset_index(drop=True)
        tables['best_topk'] = register_table('best_topk_models', best_topk)
    return tables

#best_model_tables = build_best_model_tables(metrics_df)
#best_ranking_tables = build_best_ranking_tables()

## 13. Current PnL-specific temporal decile analysis

This section is separate because current PnL is a continuous trading-outcome target and benefits from temporal decile diagnostics. The purpose is to test whether higher model-predicted scores correspond to better realised current PnL over time.

In [41]:
def predicted_decile_outcome_table(pred_df, n_deciles=10, min_rows=30):
    if pred_df is None or len(pred_df) < min_rows: return pd.DataFrame()
    df = pred_df.dropna(subset=[TRUE_COL, SCORE_COL]).copy()
    if len(df) < min_rows: return pd.DataFrame()
    df['prediction_rank_pct'] = df[SCORE_COL].rank(method='first', pct=True)
    df['prediction_decile'] = np.ceil(df['prediction_rank_pct'] * n_deciles).clip(1, n_deciles).astype(int)
    return df.groupby('prediction_decile').agg(n=(TRUE_COL, 'size'), mean_true=(TRUE_COL, 'mean'), median_true=(TRUE_COL, 'median'), std_true=(TRUE_COL, 'std'), mean_score=(SCORE_COL, 'mean'), positive_true_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') > 0).mean()), negative_true_rate=(TRUE_COL, lambda x: (pd.to_numeric(x, errors='coerce') < 0).mean())).reset_index()


def temporal_decile_analysis(predictions_df, target_name='rl_long_current_pnl', n_deciles=10):
    if predictions_df is None or len(predictions_df) == 0: return pd.DataFrame(), pd.DataFrame()
    df = predictions_df[predictions_df['target_name'] == target_name].copy()
    if len(df) == 0: return pd.DataFrame(), pd.DataFrame()
    group_cols = ['target_name', 'feature_set', 'model', SPLIT_COL]
    detail_parts = []; year_parts = []
    for keys, g in df.groupby(group_cols, dropna=False):
        detail = predicted_decile_outcome_table(g, n_deciles=n_deciles)
        if len(detail):
            for col, value in zip(group_cols, keys): detail[col] = value
            detail_parts.append(detail)
        if YEAR_COL in g.columns:
            for year, gy in g.groupby(YEAR_COL, dropna=False):
                yd = predicted_decile_outcome_table(gy, n_deciles=n_deciles, min_rows=20)
                if len(yd):
                    for col, value in zip(group_cols, keys): yd[col] = value
                    yd[YEAR_COL] = year; year_parts.append(yd)
    detail_df = pd.concat(detail_parts, ignore_index=True, sort=False) if detail_parts else pd.DataFrame()
    year_df = pd.concat(year_parts, ignore_index=True, sort=False) if year_parts else pd.DataFrame()
    register_table('current_pnl_prediction_deciles', detail_df)
    register_table('current_pnl_prediction_deciles_by_year', year_df)
    return detail_df, year_df

#current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(predictions_df)

## 14. Optional grouped-model experiments

This section supports cases where one global panel model may be misleading because tickers or industries have very different label distributions. It trains separate models by group, such as ticker or SIC2, and exports group-level performance tables.

In [ ]:
GROUPED_MODEL_CONFIG = {
    'enabled': False,
    'group_columns': [TICKER_COL, SIC2_COL],
    'target_names': ['rl_long_current_pnl'],
    'feature_set_names': ['combined_project_b'],
    'min_train_rows_per_group': 100,
    'min_eval_rows_per_group': 30,
}


def run_grouped_model_experiments(frames, feature_sets, target_configs=TARGET_CONFIGS, grouped_config=GROUPED_MODEL_CONFIG):
    if not grouped_config.get('enabled', False):
        print('Grouped model experiments disabled.'); return pd.DataFrame(), pd.DataFrame()
    metric_rows = []; prediction_parts = []
    for group_col in grouped_config['group_columns']:
        if group_col not in frames['train'].columns: continue
        for target_name in grouped_config['target_names']:
            if target_name not in target_configs: continue
            target_cfg = target_configs[target_name]; target_col = target_cfg['column']; task = target_cfg['task']
            if target_col not in frames['train'].columns: continue
            models = get_models_for_task(task)
            for feature_set_name in grouped_config['feature_set_names']:
                if feature_set_name not in feature_sets: continue
                feature_cols = feature_sets[feature_set_name]
                for group_value in frames['train'][group_col].dropna().unique():
                    train_base = filter_rows_for_target(frames["train"], target_name)
                    valid_base = filter_rows_for_target(frames["valid"], target_name)
                    test_base = filter_rows_for_target(frames["test"], target_name)
                    train_g = train_base[train_base[group_col] == group_value].copy()
                    X_train, y_train = prepare_xy(train_g, feature_cols, target_col, task)
                    if len(y_train) < grouped_config['min_train_rows_per_group']: continue
                    if task in ['binary', 'multiclass'] and y_train.nunique(dropna=True) < 2: continue
                    for model_name, model in models.items():
                        if model_name.startswith('Dummy'): continue
                        fitted = clone(model)
                        try: fitted.fit(X_train, y_train)
                        except Exception: continue
                        for split_name in ['valid', 'test']:
                            eval_g = frames[split_name][frames[split_name][group_col] == group_value].copy()
                            X_eval, y_eval = prepare_xy(eval_g, feature_cols, target_col, task)
                            if len(y_eval) < grouped_config['min_eval_rows_per_group']: continue
                            y_pred = fitted.predict(X_eval)
                            if task == 'binary':
                                y_score = get_positive_proba(fitted, X_eval)
                                if y_score is None: y_score = y_pred
                                metrics = binary_metrics(y_eval, y_pred, y_score)
                            elif task == 'regression':
                                y_score = y_pred; metrics = regression_metrics(y_eval, y_pred)
                            else:
                                y_score = y_pred; metrics = multiclass_metrics(y_eval, y_pred)
                            row = {'group_col': group_col, 'group_value': group_value, 'target_name': target_name, 'task': task, 'feature_set': feature_set_name, 'model': model_name, 'split': split_name, 'n_features': len(feature_cols)}
                            row.update(metrics); metric_rows.append(row)
                            pred_df = build_prediction_frame(eval_g, y_eval, y_pred, y_score, target_name, task, feature_set_name, f'grouped_{group_col}_{model_name}', split_name)
                            pred_df['group_model_col'] = group_col; pred_df['group_model_value'] = group_value
                            prediction_parts.append(pred_df)
    grouped_metrics = pd.DataFrame(metric_rows)
    grouped_predictions = pd.concat(prediction_parts, ignore_index=True, sort=False) if prediction_parts else pd.DataFrame()
    register_table('grouped_model_metrics', grouped_metrics)
    if len(grouped_predictions) > 0:
        path = PREDICTION_DIR / f'grouped_model_predictions_{RUN_TIMESTAMP}.parquet'
        grouped_predictions.to_parquet(path, index=False)
        print('Grouped prediction parquet exported:', path)
    return grouped_metrics, grouped_predictions

GROUPED_MODEL_CONFIG['enabled'] = True
#grouped_metrics, grouped_predictions = run_grouped_model_experiments(frames, FEATURE_SET_CONFIGS)

## 15. One-click execution block

This section provides the intended full workflow. After updating `DATASET_CONFIG`, run this cell to execute the integrated experiment pipeline. For a large dataset, start with a small subset of targets/features/models first, then expand after confirming the pipeline runs correctly.

In [ ]:
# Recommended first test run:
EXPERIMENT_CONFIG['target_names'] = ['pi_hindsight_entry_long','pi_hindsight_entry_original', 
                                        'rl_expert_action','rl_long_action_level',
                                        'rl_long_reward', 'rl_long_current_pnl',
                                        'rl_long_reward_margin', 'rl_long_is_best']
EXPERIMENT_CONFIG['feature_set_names'] = ['combined_project_b', 'momentum_volatility_only', 'fundamentals_only', 'calc_algo_only']
EXPERIMENT_CONFIG['model_names'] = ['DummyMean','Logistic', 'Ridge', 'Lasso', 'ElasticNet', 'RandomForest', 'LightGBM']

# Full workflow:
frames, all_data = load_dataset_from_config(DATASET_CONFIG)
frames, all_data = apply_target_construction(frames)
label_eda_tables = run_all_label_eda(all_data)
FEATURE_SET_CONFIGS = build_feature_sets(all_data, DATASET_CONFIG)
metrics_df, predictions_df, skipped_df = run_integrated_experiments(frames, FEATURE_SET_CONFIGS)


Parquet cache found. Loading parquet instead of JSONL:
  c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_train.parquet
Loaded dataframe shape: (98419, 748)
train: (98419, 750)
Parquet cache found. Loading parquet instead of JSONL:
  c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_validation.parquet
Loaded dataframe shape: (24469, 748)
valid: (24469, 750)
Parquet cache found. Loading parquet instead of JSONL:
  c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\universe_100_recent_post_normalisation_test.parquet
Loaded dataframe shape: (32142, 748)
test: (32142, 750)
Running: target=pi_hindsight_entry_long | features=combined_project_b | model=DummyMean
Running: target=pi_hindsight_entry_long | features=combined_project_b | model=Ridge
Running: target=pi_hindsight_entry_long | features=combined_project_b | model=Lasso
Running: target=pi

NameError: name 'SIC_COL' is not defined

In [48]:
diagnostic_tables = run_prediction_diagnostics(predictions_df)
best_model_tables = build_best_model_tables(metrics_df)
best_ranking_tables = build_best_ranking_tables()
current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(predictions_df)
grouped_metrics, grouped_predictions = run_grouped_model_experiments(frames, FEATURE_SET_CONFIGS)
final_workbook_path = export_registered_tables()

Grouped prediction parquet exported: c:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\integrated_experiment_outputs\predictions\grouped_model_predictions_20260618_193353.parquet


ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.

In [50]:
final_workbook_path = export_registered_tables()

IndexError: At least one sheet must be visible

In [53]:
predictions_df = pd.read_parquet(
    r"C:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\integrated_experiment_outputs\predictions\integrated_predictions_20260618_193353.parquet"
)

print(predictions_df.shape)
predictions_df.head()
print(predictions_df.columns.tolist())

def rebuild_metrics_from_predictions(predictions_df):
    rows = []

    group_cols = [
        "target_name",
        "task",
        "feature_set",
        "model",
        SPLIT_COL,
    ]

    for keys, g in predictions_df.groupby(group_cols, dropna=False):
        target_name, task, feature_set, model_name, split_name = keys

        y_true = g[TRUE_COL]
        y_pred = g[PRED_COL]
        y_score = g[SCORE_COL] if SCORE_COL in g.columns else None

        if task == "regression":
            metrics = regression_metrics(y_true, y_pred)

        elif task == "binary":
            metrics = binary_metrics(y_true, y_pred, y_score)

        elif task == "multiclass":
            metrics = multiclass_metrics(y_true, y_pred)

        else:
            continue

        row = {
            "target_name": target_name,
            "task": task,
            "feature_set": feature_set,
            "model": model_name,
            SPLIT_COL: split_name,
            "n_rows": len(g),
        }

        row.update(metrics)
        rows.append(row)

    metrics_df = pd.DataFrame(rows)

    register_table("model_metric_summary_rebuilt", metrics_df)

    return metrics_df

metrics_df = rebuild_metrics_from_predictions(predictions_df)
metrics_df.head()

diagnostic_tables = run_prediction_diagnostics(predictions_df)

best_model_tables = build_best_model_tables(metrics_df)

best_ranking_tables = build_best_ranking_tables()

current_pnl_deciles, current_pnl_deciles_by_year = temporal_decile_analysis(
    predictions_df
)

final_workbook_path = export_registered_tables()

(5434656, 12)
['attr__timestamp', 'year', 'attr__ticker', 'attr__sic2', 'target_name', 'task', 'feature_set', 'model', 'split', 'y_true', 'y_pred', 'prediction_score']
Integrated workbook exported to: C:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\integrated_experiment_outputs\tables\integrated_experiment_tables_20260618_193353.xlsx
Number of registered tables: 17


In [ ]:
grouped_predictions_df = pd.read_parquet(
    r"C:\Users\user\Downloads\universe_100_recent_post_normalisation_experiment\integrated_experiment_outputs\predictions\grouped_model_predictions_20260618_193353.parquet"
)

print(grouped_predictions_df.shape)
grouped_predictions_df.head()
grouped_diagnostic_tables = run_prediction_diagnostics(grouped_predictions_df)

register_table(
    "grouped_predictions_sample",
    grouped_predictions_df.head(50000)
)

final_workbook_path = export_registered_tables()

In [54]:
print("Number of registered tables:", len(EXPORTED_TABLES))

for name, df in EXPORTED_TABLES.items():
    print(name, None if df is None else df.shape)

Number of registered tables: 17
model_metric_summary_rebuilt (192, 32)
daily_spearman_detail (55776, 10)
daily_spearman_summary (192, 9)
topk_detail (111552, 17)
topk_summary (384, 13)
within_ticker_detail (18816, 10)
within_ticker_summary (192, 9)
subgroup_diagnostics_by_year (288, 30)
subgroup_diagnostics_by_attr_ticker (18816, 30)
subgroup_diagnostics_by_attr_sic2 (5760, 30)
binary_calibration_tables (360, 10)
best_model_by_target (14, 33)
feature_family_comparison (42, 7)
best_daily_spearman_models (14, 9)
best_topk_models (10, 13)
current_pnl_prediction_deciles (360, 12)
current_pnl_prediction_deciles_by_year (540, 13)


In [ ]:
frames, all_data = load_dataset_from_config(DATASET_CONFIG)
frames, all_data = apply_target_construction(frames)
column_inventory = build_column_inventory_tables(frames, all_data, OUTPUT_DIR)
rl_trainable_eda_tables = run_rl_trainable_eda(all_data)

label_eda_tables = run_all_label_eda(all_data)

FEATURE_SET_CONFIGS = build_feature_sets(all_data, DATASET_CONFIG)

## 16. Checklist

This final section creates a checklist of all tasks covered by the notebook. It is exported as part of the Excel outputs so the final results can be audited and traced back to the workflow.

In [55]:
def build_notebook_checklist():
    rows = [
        ('1. Imports and global settings', 'Defined imports, random seed, canonical column names, and output directories.', 'Keeps variables consistent across the whole notebook.'),
        ('2. Dataset configuration', 'Dataset paths and dataset-level differences are controlled from DATASET_CONFIG.', 'Allows Universe/time horizon/pre-post changes without rewriting modelling code.'),
        ('3. Data loading', 'Created loaders for parquet/csv/json/jsonl and standardised train/valid/test frames.', 'Ensures all downstream functions receive consistent DataFrames.'),
        ('4. Export manager', 'Created register_table and export_registered_tables.', 'Ensures all EDA, metrics, diagnostics, and comparison tables are exported to Excel.'),
        ('5. Label construction', 'Added derived target columns for pi_hindsight and RL long-side labels.', 'Makes label definitions explicit and reusable.'),
        ('6. Label EDA', 'Created overall, split, year, ticker, and SIC2 label EDA tables.', 'Checks sparsity, split stability, time variation, and ticker heterogeneity.'),
        ('7. Feature sets', 'Created Project B, all-numeric, fundamentals, valuation, momentum/volatility, macro, and algorithmic feature sets.', 'Allows fair feature-family ablation and comparison.'),
        ('8. Modelling utilities', 'Created shared preparation/prediction functions plus task-specific model dictionaries.', 'Reuses common code while separating regression/binary/multiclass logic.'),
        ('9. Metrics', 'Created regression, binary, multiclass, calibration, daily ranking, Top-K, within-ticker, and subgroup metrics.', 'Evaluates both statistical prediction and practical ranking usefulness.'),
        ('10. Main runner', 'Created integrated loop over targets, feature sets, and models.', 'Runs most experiments from one consistent framework.'),
        ('11. Post-model diagnostics', 'Created daily Spearman, Top-K, within-ticker, by-year, by-ticker, by-sector, and calibration diagnostics.', 'Reveals whether weak average performance hides useful subgroup or ranking behaviour.'),
        ('12. Best-model tables', 'Created best-model, feature-family, best-ranking, and best-Top-K comparison tables.', 'Supports concise dissertation interpretation.'),
        ('13. Current PnL temporal deciles', 'Created current PnL prediction-decile analysis overall and by year.', 'Connects model scores to realised trading outcome quality.'),
        ('14. Optional grouped models', 'Added optional ticker/SIC2 grouped-model framework.', 'Tests whether separate group models are better than one global panel model.'),
        ('15. One-click execution', 'Provided a full execution block for the integrated workflow.', 'Makes the notebook easier to rerun and audit.'),
    ]
    checklist = pd.DataFrame(rows, columns=['section', 'task_completed', 'why_it_matters'])
    register_table('notebook_task_checklist', checklist)
    return checklist

notebook_task_checklist = build_notebook_checklist()
notebook_task_checklist

,section,task_completed,why_it_matters
0,1. Imports and global settings,"Defined imports, random seed, canonical column...",Keeps variables consistent across the whole no...
1,2. Dataset configuration,Dataset paths and dataset-level differences ar...,Allows Universe/time horizon/pre-post changes ...
2,3. Data loading,Created loaders for parquet/csv/json/jsonl and...,Ensures all downstream functions receive consi...
3,4. Export manager,Created register_table and export_registered_t...,"Ensures all EDA, metrics, diagnostics, and com..."
4,5. Label construction,Added derived target columns for pi_hindsight ...,Makes label definitions explicit and reusable.
5,6. Label EDA,"Created overall, split, year, ticker, and SIC2...","Checks sparsity, split stability, time variati..."
6,7. Feature sets,"Created Project B, all-numeric, fundamentals, ...",Allows fair feature-family ablation and compar...
7,8. Modelling utilities,Created shared preparation/prediction function...,Reuses common code while separating regression...
8,9. Metrics,"Created regression, binary, multiclass, calibr...",Evaluates both statistical prediction and prac...
9,10. Main runner,"Created integrated loop over targets, feature ...",Runs most experiments from one consistent fram...


## 17. Final Excel export

Run this after completing the EDA and modelling workflow. It exports all registered tables into one integrated workbook and also creates individual Excel files for each registered DataFrame.

In [45]:
final_workbook_path = export_registered_tables(
    workbook_name=f'integrated_experiment_tables_{RUN_TIMESTAMP}.xlsx',
    export_individual_files=True,
)
final_workbook_path

ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.

## Optional Robustness Check: Rolling / Expanding Window Evaluation